In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced, SampleOutcomesAdvancedPCR

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
model_name = 'pcr'

n_processes = 32

log_name = 'train'
with open('../transformed_event_logs/PCR_start_end_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

#test_event_log['time:timestamp'] = test_event_log['time:timestamp_complete']
test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_activities = ['Callback timeout', 'Export result', 'Export to EMS', 'Match patient data', 'Receive sample state', 'Send notification', 'Wait for plate validation', 'timeout']

/tmp/ipykernel_2444399/396940377.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_event_log = pickle.load(f)


In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_seconds-in-day/'
evaluator_CS = ConductEvaluation(drbart_model_path, SampleOutcomesAdvancedPCR, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'timestamp_key' : 'time:timestamp_start',
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_CS = evaluator_CS.sample_cases(False, multiprocessing=True)

100%|██████████| 4927/4927 [00:17<00:00, 289.78it/s]


In [4]:
with open('test.pickle', 'wb') as f:
    pickle.dump(likelihoods_CS, f)

In [5]:
with open('test.pickle', 'rb') as f:
   likelihoods_CS = pickle.load(f)

In [6]:
np.mean([v.ln() for v in likelihoods_CS[0].values()])

Decimal('-Infinity')

In [7]:
np.mean(get_pscores(likelihoods_CS))

np.float64(15169.019500124743)

In [8]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_seconds-in-day_day-of-week/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvancedPCR, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'timestamp_key' : 'time:timestamp_start',
                                                        'categorical_args' : ['concept_name', 'day_of_week'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, multiprocessing=True)

100%|██████████| 4927/4927 [00:16<00:00, 296.98it/s]


In [9]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [10]:
np.mean(get_pscores(likelihoods_A))

np.float64(11080.390897092246)